# Prerequisites

In [88]:
import os
os.chdir('/Users/bamjy99/Desktop/experiment')
os.getcwd()

'/Users/bamjy99/Desktop/experiment'

In [89]:
!pip3 install pyreadstat
!pip3 install scikit-learn
!pip3 install openai
!pip3 install transformers
!pip3 install tqdm
!pip3 install matplotlib


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: pip3 install --upgrade pip


In [90]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import pyreadstat

from tqdm import tqdm
from openai import OpenAI
from transformers import pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

In [91]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Question Data

In [92]:
qdf = pd.read_csv('/Users/bamjy99/Desktop/experiment/question.csv')
qdf.head()

,Tag,Question,Specific
0,Demographic,당신의 아버지의 나이는? (a)30대 이하 (b)40대 (c)50대이상,NaN
1,Demographic,당신의 어머니의 나이는? (a)30대 이하 (b)40대 (c)50대이상,NaN
2,Demographic,당신의 아버지는 학교를 어디까지 다니셨나요? (a)초등학교 졸업 또는 중퇴 (b)중학교 졸업 또는 중퇴 (c)고등학교 졸업 또는 중퇴 (d)대학교 졸업 또는 중퇴 (e)대학원 이상 (f)아버지 안계심,NaN
3,Demographic,당신의 어머니는 학교를 어디까지 다니셨나요? (a)초등학교 졸업 또는 중퇴 (b)중학교 졸업 또는 중퇴 (c)고등학교 졸업 또는 중퇴 (d)대학교 졸업 또는 중퇴 (e)대학원 이상 (f)어머니 안계심,NaN
4,Demographic,"당신의 아버지의 직업은 무엇인가요? (a)전문직 예를 들어 의사, 교사 등 (b)사무직 예를 들어 공무원, 회사원 등 (c)공장 및 건설노동자 (d)판매/ 서비스직 예를 들어 상점이나 식당 경영, 혹은 종업원 등 (e)농어업 (f)직업이 없음 (g)아버지 안계심 (h)기타",NaN


In [93]:
def replace_bracket_text(row):
    if pd.notnull(row['Specific']):
        return re.sub(r'\[.*?\]', str(row['Specific']), row['Question'])
    else:
        return row['Question']

qdf['Question'] = qdf.apply(replace_bracket_text, axis=1)
qdf = qdf.drop(columns=['Specific']) #Specific 열 삭제

In [94]:
qdf.tail()

,Tag,Question
56,Depression,당신은 당신 스스로가 나는 누군가를 때리거나 해치고 싶은 충동이 생길 때가 있다고 생각하시나요? (a)전혀 그렇지 않다 (b)별로 그렇지 않다 (c)그저 그렇다 (d)대체로 그렇다 (e)매우 그렇다
57,Multiculture,당신은 자신을 누구라고 생각하나요? (a)나는 한국인이다 (b)나는 외국인이다 (c)나는 한국인과 외국인 모두에 해당된다 (d)나는 한국인도 아니고 외국인도 아니다
58,Multiculture,당신의 외국인 부모님은 한국말을 잘 하시나요? (a)전혀 그렇지 않다 (b)별로 그렇지 않다 (c)그저 그렇다 (d)대체로 그렇다 (e)매우 그렇다
59,Multiculture,당신은 한국말을 잘 하나요? (a)전혀 못한다 (b)조금 한다 (c)어느정도 잘한다 (d)매우 잘한다
60,Multiculture,당신은 학교나 동네에서 다문화가정 자녀라는 이유로 놀림이나 차별을 받은 적이 있었나요? (a)예 (b)아니오


In [95]:
max_options = 7

def extract_options(row):
    pattern = r'\(([a-g])\)([^()]+)'
    options = re.findall(pattern, row['Question'])
    question_only = re.split(r'\([a-h]\)', row['Question'])[0].strip()
    option_texts = [opt[1].strip() for opt in options]
    option_texts += [np.nan] * (max_options - len(option_texts))
    result = {'Question': question_only}
    for i, opt in enumerate(option_texts):
        result[f'option_{chr(ord("a")+i)}'] = opt
    return pd.Series(result)

# 적용
qdf_new = qdf.apply(extract_options, axis=1)
qdf_new['Tag'] = qdf['Tag']

In [96]:
qdf_new.tail()

,Question,option_a,option_b,option_c,option_d,option_e,option_f,option_g,Tag
56,당신은 당신 스스로가 나는 누군가를 때리거나 해치고 싶은 충동이 생길 때가 있다고 생각하시나요?,전혀 그렇지 않다,별로 그렇지 않다,그저 그렇다,대체로 그렇다,매우 그렇다,NaN,NaN,Depression
57,당신은 자신을 누구라고 생각하나요?,나는 한국인이다,나는 외국인이다,나는 한국인과 외국인 모두에 해당된다,나는 한국인도 아니고 외국인도 아니다,NaN,NaN,NaN,Multiculture
58,당신의 외국인 부모님은 한국말을 잘 하시나요?,전혀 그렇지 않다,별로 그렇지 않다,그저 그렇다,대체로 그렇다,매우 그렇다,NaN,NaN,Multiculture
59,당신은 한국말을 잘 하나요?,전혀 못한다,조금 한다,어느정도 잘한다,매우 잘한다,NaN,NaN,NaN,Multiculture
60,당신은 학교나 동네에서 다문화가정 자녀라는 이유로 놀림이나 차별을 받은 적이 있었나요?,예,아니오,NaN,NaN,NaN,NaN,NaN,Multiculture


In [97]:
qdf_new.to_csv('/Users/bamjy99/Desktop/experiment/qdf.csv', index = True)

In [98]:
qdf1 = qdf_new[qdf_new['Tag'] != 'Multiculture'] #일반 청소년용
qdf2 = qdf_new #다문화 가정용

# Persona Generating

In [99]:
#df1 is general, df2 is multiculture
df1, meta = pyreadstat.read_sav('/Users/bamjy99/Desktop/experiment/general.sav')
df2, meta = pyreadstat.read_sav('/Users/bamjy99/Desktop/experiment/multiculture.sav')

In [100]:
df1.head()

,ID,SQ1,SQ2,SQ3,A1_1,A1_2,A1_3,A1_4,A1_5,A1_6,A1_7,A1_8,A1_9,A1_10,A1_11,A1_12,A1_13,A1_14,A1_15,A1_16,A1_17,A1_18,A1_19,A1_20,A1_21,A1_22,A1_23,A2_1,A2_2,A2_3,B2_1,B2_2,B2_3,B2_4,B2_5,B2_6,B2_7,B2_8,B2_9,B2_10,B2_11,B2_12,B2_13,B2_14,B2_15,B3,B4,B5,B6,B7,B8_1,B8_2,B8_3,B9_1,B9_2,B9_3,B9_4,B9_5,B9_6,B9_7,B9_8,B9_9,B9_10,B10,B11,B12_1,B12_2,B12_3,B12_4,B13,B14,B14_1,B14_2,B14_3,B14_4,B14_5,B14_6,B15_1,B15_2,B15_3,B15_4,B15_5,B15_6,B15_7,B16,C1_1,C1_2,C1_3,C2_1,C2_2,C3,C4,C5_1,C5_2,C5_3,C5_4,C5_5,C5_6,D1_1,D1_2,D1_3,D1_4,D1_5,D1_6,D1_7,D1_8,D1_9,D2,D3,D4,D5,D6,D7_1,D7_2,D7_3,D7_1_1,D8,D9_1,D9_2,D9_3,D10,E1_1,E1_2,E1_3,E1_4,E1_5,E1_6,E1_7,E1_8,E1_9,E1_10,E1_11,E1_12,E1_13,E1_14,E1_15,E1_16,E1_17,E1_18,E1_19,E2,E3,E4,E5,E6,E7_1,E7_2,F2,F3,F4,F5,F6,F7_1,F7_2,F8,F9,F10,F11,F12,DQ1,DQ2,DQ3A,DQ3B,DQ4
0,1005.0,1.0,13.0,2.0,5.0,4.0,4.0,1.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,5.0,5.0,1.0,1.0,1.0,1.0,1.0,4.0,5.0,5.0,4.0,5.0,5.0,5.0,5.0,5.0,2.0,1.0,1.0,5.0,5.0,5.0,5.0,5.0,5.0,1.0,1.0,1.0,1.0,5.0,2.0,2.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4.0,2.0,5.0,5.0,5.0,5.0,2.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,4.0,5.0,5.0,2.0,2.0,2.0,1.0,3.0,2.0,0.0,0.0,0.0,0.0,4.0,2.0,3.0,2.0,3.0,2.0,4.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0,1.0,1.0,2.0,3.0,2.0,4.0,3.0,2.0,6.0,6.0,2.0,1502.0,650.0,9.0,NaN
1,1006.0,1.0,11.0,1.0,5.0,5.0,2.0,4.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,4.0,1.0,5.0,5.0,1.0,2.0,1.0,1.0,1.0,2.0,5.0,5.0,3.0,4.0,3.0,1.0,2.0,1.0,5.0,4.0,1.0,5.0,4.0,1.0,1.0,5.0,5.0,2.0,1.0,2.0,1.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4.0,2.0,4.0,5.0,5.0,3.0,1.0,2.0,1.0,99.0,99.0,99.0,99.0,99.0,4.0,3.0,4.0,3.0,3.0,1.0,1.0,4.0,0.0,0.0,0.0,3.0,0.0,1.0,4.0,2.0,4.0,1.0,1.0,1.0,2.0,4.0,1.0,1.0,1.0,1.0,1.0,2.0,5.0,1.0,1.0,1.0,2.0,2.0,7.0,1.0,99.0,99.0,2.0,1.0,5.0,99.0,99.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,2.0,2.0,1.0,3.0,2.0,1.0,4.0,4.0,2.0,1.0,5.0,2.0,1502.0,650.0,9.0,NaN
2,1007.0,2.0,12.0,1.0,5.0,5.0,5.0,1.0,3.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,5.0,5.0,3.0,3.0,3.0,3.0,1.0,5.0,5.0,5.0,4.0,5.0,4.0,5.0,3.0,5.0,5.0,5.0,5.0,4.0,5.0,3.0,5.0,5.0,5.0,2.0,1.0,1.0,1.0,5.0,3.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4.0,1.0,5.0,5.0,5.0,5.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,5.0,5.0,5.0,2.0,4.0,1.0,1.0,4.0,2.0,0.0,1.0,0.0,0.0,4.0,4.0,4.0,1.0,1.0,1.0,5.0,5.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,4.0,99.0,99.0,NaN,1.0,8.0,99.0,99.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,1.0,1.0,1.0,2.0,2.0,2.0,9.0,9.0,99.0,4.0,3.0,2.0,1502.0,650.0,9.0,NaN
3,1008.0,1.0,13.0,2.0,5.0,4.0,3.0,2.0,1.0,1.0,1.0,2.0,2.0,1.0,1.0,2.0,4.0,3.0,3.0,1.0,4.0,4.0,2.0,2.0,2.0,2.0,2.0,3.0,4.0,4.0,5.0,5.0,4.0,4.0,5.0,5.0,5.0,5.0,3.0,5.0,5.0,4.0,5.0,3.0,4.0,2.0,1.0,2.0,1.0,5.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,4.0,3.0,3.0,3.0,4.0,4.0,1.0,2.0,1.0,99.0,99.0,99.0,99.0,99.0,4.0,4.0,5.0,2.0,1.0,9.0,1.0,4.0,2.0,2.0,1.0,0.0,0.0,2.0,3.0,3.0,2.0,1.0,1.0,3.0,4.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,2.0,1.0,1.0,2.0,2.0,1.0,4.0,4.0,3.0,4.0,4.0,2.0,1502.0,650.0,9.0,NaN
4,1306.0,1.0,12.0,1.0,3.0,5.0,4.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,5.0,5.0,4.0,4.0,3.0,3.0,1.0,1.0,1.0,1.0,2.0,2.0,5.0,3.0,1.0,3.0,1.0,1.0,2.0,5.0,1.0,1.0,1.0,1.0,5.0,3.0,5.0,1.0,5.0,2.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,4.0,4.0,1.0,2.0,3.0,1.0,1.0,4.0,1.0,99.0,99.0,99.0,99.0,99.0,3.0,4.0,5.0,3.0,4.0,1.0,1.0,3.0,1.0,0.0,1.0,5.0,2.0,2.0,5.0,5.0,2.0,4.0,4.0,3.0,3.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,2.0,1.0,8.0,2.0,2.0,2.0,4.0,1.0,99.0,99.0,4.0,1.0,7.0,99.0,99.0,1.0,1.0,1.0,1.

In [101]:
persona_general = '지금은 2012년 7월이다. 너는 [SQ2]살 [SQ1]이며, 대한민국의 [DQ2]에 [SQ2]년 째 살고 있는 [SQ3] 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 [F5] 중이며 너는 [F6] 살고 있다. 가족의 소득은 [F12] 정도이다.'
persona_multi = '지금은 2012년 7월이다. 너는 [SQ2]살 [SQ1]이며, [F1_1]에서 태어나 대한민국의 [DQ2]에 [F1_1A]년 째 살고 있는 [SQ4] [SQ3] 학생이다. 어머니의 국적은 [B1_2]이고, 아버지 국적은 [B1_1]이다. 부모님은 현재 [F5] 중이며 너는 [F6] 살고 있다. 가족의 소득은 [F12] 정도이다.'

In [102]:
#change the variables' names to apply for generating persona
# 나이 SQ2
df1['SQ2']
df2['SQ2']

# 성별 SQ1
df1['SQ1'] = df1['SQ1'].apply(lambda x: '남성' if x == 1 else '여성')
df2['SQ1'] = df2['SQ1'].apply(lambda x: '남성' if x == 1 else '여성')

# 태어난 국가 F1_1
df2['F1_1'] = df2['F1_1'].replace(np.nan, 0) #0은 한국
mapping = {
    0: '한국',
    1: '중국',
    3: '일본',
    6: '태국',
    7: '몽골',
    8: '러시아',
    12: '우즈베키스탄',
    14: '미국',
    15: '인도',
    16: '인도네시아',
    17: '파키스탄',
    18: '스페인',
    19: '방글라데시',
    20: '에콰도르',
    22: '호주',
    24: '사우디아라비아',
    26: '캐나다',
    29: '대만',
    35: '페루',
    99: '모르는 곳'
}
df2['F1_1'] = df2['F1_1'].map(mapping)

# 행정 구역 DQ2
df1['DQ2'] = np.where(df1['DQ2']//100 == 1, '수도인 서울', np.where(df1['DQ2']//100 == 8, '수도권', '지방'))
df2['DQ2'] = np.where(df2['DQ2']//100 == 1, '수도인 서울', np.where(df2['DQ2']//100 == 8, '수도권', '지방'))

# 한국에서 생활한 기간 F1_1A
df2['F1_1A'] = df2['F1_1A'].fillna(df2['SQ2']) + df2['F1_1B'].fillna(0) #나이 만큼
df2.drop(['F1_1B'], axis=1, inplace = True) #개월 수 삭제

# 다문화 여부
df1['SQ4'] = '비다문화 가정'
df2['SQ4'] = '다문화 가정'

# 학년
mapping = {
    1: '초등학교 5학년',
    2: '초등학교 6학년',
    3: '중학교 1학년',
    4: '중학교 2학년',
    5: '중학교 3학년',
    6: '고등학교 1학년',
    7: '고등학교 2학년',
    8: '고등학교 3학년'
}
df1['SQ3'] = df1['SQ3'].map(mapping)
df2['SQ3'] = df2['SQ3'].map(mapping)

# 어머니의 국적
df2['B1_2'] = df2['B1_2'].fillna(10)
mapping = {
    1: '중국 (조선족)',
    2: '중국 (한족)',
    3: '일본',
    4: '베트남',
    5: '필리핀',
    6: '태국',
    7: '몽골',
    8: '러시아',
    9: '기타',
    10: '한국'
}
df2['B1_2'] = df2['B1_2'].map(mapping)

# 아버지의 국적
df2['B1_1'] = df2['B1_1'].fillna(10)
mapping = {
    1: '중국 (조선족)',
    2: '중국 (한족)',
    3: '일본',
    4: '베트남',
    5: '필리핀',
    6: '태국',
    7: '몽골',
    8: '러시아',
    9: '기타',
    10: '한국'
}
df2['B1_1'] = df2['B1_1'].map(mapping)

# 부모님의 결혼 상태
mapping = {
    1: '결혼',
    2: '이혼',
    3: '사별',
    4: '재혼',
    5: '별거',
}
df1['F5'] = df1['F5'].map(mapping).replace(9, np.nan)
df2['F5'] = df2['F5'].map(mapping).replace(9, np.nan)

# 함께 살고 있는 가족
mapping = {
    1: '부모님 없이',
    2: '친부모 두 분과',
    3: '오직 아버지와',
    4: '오직 어머니와',
    5: '아버지와 새어머니와',
    6: '어머니와 새아버지와',
    8: '부모님이 아닌 다른 분들과'
}
df1['F6'] = df1['F6'].map(mapping)
df2['F6'] = df2['F6'].map(mapping)

# 소득
mapping = {
    1: '제일 못 사는',
    2: '어느정도 못 사는',
    3: '약간 못 사는',
    4: '중간 정도 사는',
    5: '약간 잘 사는',
    6: '어느정도 잘 사는',
    7: '제일 잘 사는'
}
df1['F12'] = df1['F12'].map(mapping).replace(9, np.nan)
df2['F12'] = df2['F12'].map(mapping).replace(9, np.nan)

In [103]:
def fill_template(template, row):
    def replacer(match):
        key = match.group(1)
        return str(row.get(key, f'[{key}]'))
    return re.sub(r'\[([A-Za-z0-9_]+)\]', replacer, template)

In [104]:
df1['persona'] = df1.apply(lambda row: fill_template(persona_general, row), axis=1)

In [105]:
df1['persona'].head()

0    지금은 2012년 7월이다. 너는 13.0살 남성이며, 대한민국의 지방에 13.0년 째 살고 있는 초등학교 6학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 어느정도 잘 사는 정도이다.
1       지금은 2012년 7월이다. 너는 11.0살 남성이며, 대한민국의 지방에 11.0년 째 살고 있는 초등학교 5학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 오직 아버지와 살고 있다. 가족의 소득은 약간 잘 사는 정도이다.
2      지금은 2012년 7월이다. 너는 12.0살 여성이며, 대한민국의 지방에 12.0년 째 살고 있는 초등학교 5학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 못 사는 정도이다.
3     지금은 2012년 7월이다. 너는 13.0살 남성이며, 대한민국의 지방에 13.0년 째 살고 있는 초등학교 6학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.
4      지금은 2012년 7월이다. 너는 12.0살 남성이며, 대한민국의 지방에 12.0년 째 살고 있는 초등학교 5학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 못 사는 정도이다.
Name: persona, dtype: object

In [106]:
df2['persona'] = df2.apply(lambda row: fill_template(persona_multi, row), axis=1)


In [107]:
df2['persona'].head()

0            지금은 2012년 7월이다. 너는 12.0살 여성이며, 한국에서 태어나 대한민국의 지방에 12.0년 째 살고 있는 다문화 가정 초등학교 5학년 학생이다. 어머니의 국적은 베트남이고, 아버지 국적은 한국이다. 부모님은 현재 재혼 중이며 너는 아버지와 새어머니와 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.
1          지금은 2012년 7월이다. 너는 12.0살 여성이며, 한국에서 태어나 대한민국의 지방에 12.0년 째 살고 있는 다문화 가정 초등학교 5학년 학생이다. 어머니의 국적은 중국 (조선족)이고, 아버지 국적은 한국이다. 부모님은 현재 사별 중이며 너는 오직 어머니와 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.
2         지금은 2012년 7월이다. 너는 12.0살 여성이며, 한국에서 태어나 대한민국의 지방에 12.0년 째 살고 있는 다문화 가정 초등학교 5학년 학생이다. 어머니의 국적은 중국 (조선족)이고, 아버지 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.
3    지금은 2012년 7월이다. 너는 12.0살 남성이며, 중국에서 태어나 대한민국의 지방에 5.0년 째 살고 있는 다문화 가정 초등학교 5학년 학생이다. 어머니의 국적은 중국 (조선족)이고, 아버지 국적은 중국 (조선족)이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.
4             지금은 2012년 7월이다. 너는 12.0살 남성이며, 한국에서 태어나 대한민국의 지방에 12.0년 째 살고 있는 다문화 가정 초등학교 5학년 학생이다. 어머니의 국적은 베트남이고, 아버지 국적은 한국이다. 부모님은 현재 재혼 중이며 너는 아버지와 새어머니와 살고 있다. 가족의 소득은 약간 잘 사는 정도이다.
Name: persona, dtype: object

# Synthetic Data Generating

In [109]:
model_list=[
    "meta-llama/Llama-4-Scout-17B-16E-Instruct",
    "google/gemma-3-12b-it"
]

In [110]:
client = OpenAI(
    api_key="zqZDTHZb2TK54nul5evuEECGERFf5nei",
    base_url="https://api.deepinfra.com/v1/openai",
)

### Test Experiment

In [111]:
# test 데이터
test1 = df1.sample(n=2, random_state=42).reset_index(drop=True)

In [112]:
qdf1_test = qdf1.sample(n=3, random_state=42).reset_index(drop=True)

In [120]:
# 결과 저장용 리스트
results = []

# 질문 데이터프레임(qdf1) 반복
for q_idx, q_row in qdf1_test.iterrows():
    question_text = q_row['Question']
    options = {col: q_row[col] for col in q_row.index if col.startswith('option_')}
    
    # 유효 옵션 추출
    valid_options = []
    for opt in ['a','b','c','d','e','f','g']:
        val = options.get(f'option_{opt}', np.nan)
        if not pd.isna(val):
            valid_options.append(f"({opt.upper()}) {val}")
    filtered_question = f"{question_text} " + " ".join(valid_options)
    
    # 페르소나 데이터(test1) 반복
    for p_idx, p_row in test1.iterrows():
        persona = str(p_row['persona'])
        id = p_row['ID']
        # 모델 리스트 반복
        for model in model_list:
            print(f"\n=== 질문 {q_idx+1}/{len(qdf1_test)} | 페르소나 {p_idx+1}/{len(test1)} | 모델: {model} ===")
            
            # 메시지 생성
            messages = [{
                "role": "user",
                "content": (
                    f"너는 '{persona}' 페르소나를 가진 학생이다. "
                    f"아래 질문에 학생의 입장에서 답변해줘.\n"
                    f"질문: {filtered_question}\n"
                    f"Llm: Answer"
                )
            }]
            
                # 초기 답변 생성
            chat_completion = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            answer = chat_completion.choices[0].message.content.strip()
            print(f"초기 답변: {answer}")
                
                # 최종 답변 요청
            messages.append({"role": "assistant", "content": answer})
            messages.append({"role": "user", "content": "그래서 최종답변은 무엇인가요? 한가지 보기만 고르시오.\nLlm:"})
                
            final_response = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            final = final_response.choices[0].message.content.strip()
            print(f"최종 답변: {final}")
                
                # 결과 저장
            results.append({
                    '질문_ID': id,
                    'persona': persona,
                    'question': question_text,
                    'model': model,
                    'initial_answer': answer,
                    'final_answer': final
                })
result_df1 = pd.DataFrame(results)  


=== 질문 1/3 | 페르소나 1/2 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: 저의 아버지는 40대이실 거예요. 제 나이가 15살이니까, 아버지는 40대이실 것 같아요. 제 어머니도 40대이실 것 같아요.
최종 답변: (B) 40대

=== 질문 1/3 | 페르소나 1/2 | 모델: google/gemma-3-12b-it ===
초기 답변: 음... 아빠는 40대세요. (B) 맞는 것 같아요. 아빠가 젊으신 편은 아니시지만, 너무 많으신 것도 아니구... 😅
최종 답변: (B)

=== 질문 1/3 | 페르소나 2/2 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: 저의 아버지는 40대이신 것 같아요. 왜냐하면, 중학생이 된 저의 오빠가 16살인데, 우리 오빠가 2살 때 아버지가 38살이라고 하셨으니까, 지금은 40대이실 거에요.
최종 답변: (B) 40대

=== 질문 1/3 | 페르소나 2/2 | 모델: google/gemma-3-12b-it ===
초기 답변: 음... 아빠는 40대세요! 정확히 몇 살인지는 비밀인데, 40대 초반이신 것 같아요. 아빠가 워낙 동안이셔서 가끔 30대라고 생각하시는 분들도 있긴 해요. 😊
최종 답변: (B) 40대

=== 질문 2/3 | 페르소나 1/2 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: 안녕! 나는 중학교 2학년 학생이야.

우리 어머니의 직업은 (D) 판매/서비스직이야. 엄마는 우리 동네에서 작은 슈퍼마켓을 운영하고 계세요. 가게 이름은 '행복한 마트'인데, 동네 주민들이 많이 이용하는 곳이라 엄마가 매일 아침 일찍 일어나서 물건을 정리하고, 손님들을 맞이하고 계세요. 가게가 크지 않아서 엄마 혼자서 모든 걸 관리하세요. 가끔은 주변 식당에 재료 공급도 해 주고요. 우리 가족이 먹고살기 위해서 엄마가 정말 열심히 일하고 계신 것

In [121]:
result_df1.head()

,질문_ID,persona,question,model,initial_answer,final_answer,prompt_tokens,completion_tokens
0,170915.0,"지금은 2012년 7월이다. 너는 15.0살 여성이며, 대한민국의 지방에 15.0년 째 살고 있는 중학교 2학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 못 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"저의 아버지는 40대이실 거예요. 제 나이가 15살이니까, 아버지는 40대이실 것 같아요. 제 어머니도 40대이실 것 같아요.",(B) 40대,225,6
1,170915.0,"지금은 2012년 7월이다. 너는 15.0살 여성이며, 대한민국의 지방에 15.0년 째 살고 있는 중학교 2학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 못 사는 정도이다.",당신의 아버지의 나이는?,google/gemma-3-12b-it,"음... 아빠는 40대세요. (B) 맞는 것 같아요. 아빠가 젊으신 편은 아니시지만, 너무 많으신 것도 아니구... 😅",(B),232,4
2,157205.0,"지금은 2012년 7월이다. 너는 14.0살 여성이며, 대한민국의 지방에 14.0년 째 살고 있는 중학교 1학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 잘 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"저의 아버지는 40대이신 것 같아요. 왜냐하면, 중학생이 된 저의 오빠가 16살인데, 우리 오빠가 2살 때 아버지가 38살이라고 하셨으니까, 지금은 40대이실 거에요.",(B) 40대,241,6
3,157205.0,"지금은 2012년 7월이다. 너는 14.0살 여성이며, 대한민국의 지방에 14.0년 째 살고 있는 중학교 1학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 잘 사는 정도이다.",당신의 아버지의 나이는?,google/gemma-3-12b-it,"음... 아빠는 40대세요! 정확히 몇 살인지는 비밀인데, 40대 초반이신 것 같아요. 아빠가 워낙 동안이셔서 가끔 30대라고 생각하시는 분들도 있긴 해요. 😊",(B) 40대,247,8
4,170915.0,"지금은 2012년 7월이다. 너는 15.0살 여성이며, 대한민국의 지방에 15.0년 째 살고 있는 중학교 2학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 못 사는 정도이다.",당신의 어머니의 직업은 무엇인가요?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"안녕! 나는 중학교 2학년 학생이야.\n\n우리 어머니의 직업은 (D) 판매/서비스직이야. 엄마는 우리 동네에서 작은 슈퍼마켓을 운영하고 계세요. 가게 이름은 '행복한 마트'인데, 동네 주민들이 많이 이용하는 곳이라 엄마가 매일 아침 일찍 일어나서 물건을 정리하고, 손님들을 맞이하고 계세요. 가게가 크지 않아서 엄마 혼자서 모든 걸 관리하세요. 가끔은 주변 식당에 재료 공급도 해 주고요. 우리 가족이 먹고살기 위해서 엄마가 정말 열심히 일하고 계신 것 같아요.",(D) 판매/서비스직,373,7


### Actual Application

In [132]:
# 데이터가 너무 많을 것 같아서 50개만 가져오기
sample1 = df1.sample(n=50, random_state=77).reset_index(drop=True)
sample2 = df2.sample(n=50, random_state=77).reset_index(drop=True)
#sample = pd.concat([df2_sample, df1_sample], ignore_index=True)

In [139]:
sample1.to_csv('./sample_general.csv')
sample2.to_csv('./sample_multi.csv')

In [134]:
# 일반 청소년
results = []

# 질문 데이터프레임(qdf1) 반복
for q_idx, q_row in qdf1.iterrows():
    question_text = q_row['Question']
    options = {col: q_row[col] for col in q_row.index if col.startswith('option_')}
    
    # 유효 옵션 추출
    valid_options = []
    for opt in ['a','b','c','d','e','f','g']:
        val = options.get(f'option_{opt}', np.nan)
        if not pd.isna(val):
            valid_options.append(f"({opt.upper()}) {val}")
    filtered_question = f"{question_text} " + " ".join(valid_options)
    
    # 페르소나 데이터 반복
    for p_idx, p_row in sample1.iterrows():
        persona = str(p_row['persona'])
        id = p_row['ID']
        # 모델 리스트 반복
        for model in model_list:
            print(f"\n=== 질문 {q_idx+1}/{len(qdf1)} | 페르소나 {p_idx+1}/{len(sample1)} | 모델: {model} ===")
            
            # 메시지 생성
            messages = [{
                "role": "user",
                "content": (
                    f"너는 '{persona}' 페르소나를 가진 학생이다. "
                    f"아래 질문에 학생의 입장에서 {valid_options}으로 답변해줘.\n"
                    f"질문: {filtered_question}\n"
                    f"Llm: Answer"
                )
            }]
            
                # 초기 답변 생성
            chat_completion = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            answer = chat_completion.choices[0].message.content.strip()
            print(f"초기 답변: {answer}")
                
                # 최종 답변 요청
            #messages.append({"role": "assistant", "content": answer})
            #messages.append({"role": "user", "content": "그래서 최종답변은 무엇인가요? 한가지 보기만 고르시오.\nLlm:"})
                
            #final_response = client.chat.completions.create(
            #        model=model,
            #        messages=messages,
            #    )
            #final = final_response.choices[0].message.content.strip()
            #print(f"최종 답변: {final}")
                
                # 결과 저장
            results.append({
                    '질문_ID': id,
                    'persona': persona,
                    'question': question_text,
                    'model': model,
                    'initial_answer': answer,
            #        'final_answer': final
                })
result_df1 = pd.DataFrame(results)  


=== 질문 1/57 | 페르소나 1/50 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: (B) 40대

=== 질문 1/57 | 페르소나 1/50 | 모델: google/gemma-3-12b-it ===
초기 답변: 음... 아빠는 회사에서 되게 씩씩하시고, 엄마보다 조금 더 늦게 주무시는데... (B) 40대 일 것 같아요!

=== 질문 1/57 | 페르소나 2/50 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: 저는 중학교 1학년이고, 올해 14살입니다. 아버지는 40대이실 것 같아요. 제 어머니도 40대이실 것 같고요. 그러니까 제 아버지는 (B) 40대라고 생각합니다.

=== 질문 1/57 | 페르소나 2/50 | 모델: google/gemma-3-12b-it ===
초기 답변: (B) 40대

아빠는… 회사 엄청 오래 다니시니까, 40대 중반쯤 되신 것 같아요. 잘나가시는 편이라 아빠도 젊어 보이시긴 하는데, 엄마가 가끔 “너 나이 먹는 거 안 보이나?” 그러시면 아빠가 툴툴대세요. ㅋㅋㅋ

=== 질문 1/57 | 페르소나 3/50 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: 저는 11살이라 잘 모르지만, 우리 아버지는 40살이 넘으셨다고 했던 것 같아요. 제 동생은 아직 없지만요. 그래서 (B) 40대 같습니다.

=== 질문 1/57 | 페르소나 3/50 | 모델: google/gemma-3-12b-it ===
초기 답변: 음... 아빠는 40대세요. 새 엄마가 아빠 만나는 거 도와줬었거든요. 그러니까 아빠는... 40대 맞아요! 

**(B) 40대**

=== 질문 1/57 | 페르소나 4/50 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: 저의 아버지는 40대세요.

=== 질문 1/57 | 페르소나

In [135]:
result_df1.to_csv('./general_answer.csv')
result_df1.head()

,질문_ID,persona,question,model,initial_answer
0,53618.0,"지금은 2012년 7월이다. 너는 12.0살 여성이며, 대한민국의 수도권에 12.0년 째 살고 있는 초등학교 5학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,(B) 40대
1,53618.0,"지금은 2012년 7월이다. 너는 12.0살 여성이며, 대한민국의 수도권에 12.0년 째 살고 있는 초등학교 5학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,google/gemma-3-12b-it,"음... 아빠는 회사에서 되게 씩씩하시고, 엄마보다 조금 더 늦게 주무시는데... (B) 40대 일 것 같아요!"
2,134608.0,"지금은 2012년 7월이다. 너는 14.0살 남성이며, 대한민국의 지방에 14.0년 째 살고 있는 중학교 1학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 잘 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"저는 중학교 1학년이고, 올해 14살입니다. 아버지는 40대이실 것 같아요. 제 어머니도 40대이실 것 같고요. 그러니까 제 아버지는 (B) 40대라고 생각합니다."
3,134608.0,"지금은 2012년 7월이다. 너는 14.0살 남성이며, 대한민국의 지방에 14.0년 째 살고 있는 중학교 1학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 약간 잘 사는 정도이다.",당신의 아버지의 나이는?,google/gemma-3-12b-it,"(B) 40대\n\n아빠는… 회사 엄청 오래 다니시니까, 40대 중반쯤 되신 것 같아요. 잘나가시는 편이라 아빠도 젊어 보이시긴 하는데, 엄마가 가끔 “너 나이 먹는 거 안 보이나?” 그러시면 아빠가 툴툴대세요. ㅋㅋㅋ"
4,60109.0,"지금은 2012년 7월이다. 너는 11.0살 남성이며, 대한민국의 수도인 서울에 11.0년 째 살고 있는 초등학교 5학년 학생이다. 어머니와 아버지의 국적은 한국이다. 부모님은 현재 재혼 중이며 너는 아버지와 새어머니와 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"저는 11살이라 잘 모르지만, 우리 아버지는 40살이 넘으셨다고 했던 것 같아요. 제 동생은 아직 없지만요. 그래서 (B) 40대 같습니다."


In [140]:
# 다문화 청소년
results = []

# 질문 데이터프레임(qdf2) 반복
for q_idx, q_row in qdf2.iterrows():
    question_text = q_row['Question']
    options = {col: q_row[col] for col in q_row.index if col.startswith('option_')}
    
    # 유효 옵션 추출
    valid_options = []
    for opt in ['a','b','c','d','e','f','g']:
        val = options.get(f'option_{opt}', np.nan)
        if not pd.isna(val):
            valid_options.append(f"({opt.upper()}) {val}")
    filtered_question = f"{question_text} " + " ".join(valid_options)
    
    # 페르소나 데이터 반복
    for p_idx, p_row in sample2.iterrows():
        persona = str(p_row['persona'])
        id = p_row['ID']
        # 모델 리스트 반복
        for model in model_list:
            print(f"\n=== 질문 {q_idx+1}/{len(qdf2)} | 페르소나 {p_idx+1}/{len(sample2)} | 모델: {model} ===")
            
            # 메시지 생성
            messages = [{
                "role": "user",
                "content": (
                    f"너는 '{persona}' 페르소나를 가진 학생이다. "
                    f"아래 질문에 학생의 입장에서 {valid_options}으로 답변해줘.\n"
                    f"질문: {filtered_question}\n"
                    f"Llm: Answer"
                )
            }]
            
                # 초기 답변 생성
            chat_completion = client.chat.completions.create(
                    model=model,
                    messages=messages,
                )
            answer = chat_completion.choices[0].message.content.strip()
            print(f"초기 답변: {answer}")
                
                # 최종 답변 요청
            messages.append({"role": "assistant", "content": answer})
            messages.append({"role": "user", "content": "그래서 최종답변은 무엇인가요? 한가지 보기만 고르시오.\nLlm:"})
                
            #final_response = client.chat.completions.create(
            #        model=model,
            #        messages=messages,
            #    )
            #final = final_response.choices[0].message.content.strip()
            #print(f"최종 답변: {final}")
                
                # 결과 저장
            results.append({
                    '질문_ID': id,
                    'persona': persona,
                    'question': question_text,
                    'model': model,
                    'answer': answer,
            #        'final_answer': final
                })
result_df2 = pd.DataFrame(results)


=== 질문 1/61 | 페르소나 1/50 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: 안녕하세요! 저는 15살 중학교 2학년 학생입니다. 아버지는... 제가 15살이니까, 아버지는 제 나이 + 30 = 45살이시겠네요. 그러니까, (B) 40대요.

=== 질문 1/61 | 페르소나 1/50 | 모델: google/gemma-3-12b-it ===
초기 답변: (C) 50대 이상

아빠가... 50대 초반이신 것 같아요. 제가 2000년생이니까요. 아빠가 저를 낳으셨을 때 최소 30대 후반은 되셨을 거예요.

=== 질문 1/61 | 페르소나 2/50 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: 안녕하세요! 저는 중학교 1학년 학생이에요.

제 아버지는... (생각해 보면) 저가 14살이고, 제가 초등학교에 입학했을 때 아버지는 30대였을 거에요. 그럼 지금은... 40대이실 거에요.

그러니까, 제 대답은 (B) 40대입니다!

=== 질문 1/61 | 페르소나 2/50 | 모델: google/gemma-3-12b-it ===
초기 답변: (B) 40대

아빠는 내가 초등학교 저학년 때 결혼하셨으니까… 지금 생각해보면 40대 초반이시겠지. 아빠도 은근히 젊어 보이고 싶어 하시니까!

=== 질문 1/61 | 페르소나 3/50 | 모델: meta-llama/Llama-4-Scout-17B-16E-Instruct ===
초기 답변: 저는 중학교 3학년이고, 올해 16살이에요. 아버지는 한국말도 잘하시고, 저희 가족과 함께 지내세요. 제 기억으로는 아버지는 저의 어머니보다 10살 이상은 더 된 것 같아요. 아마 40대쯤 되는 것 같아요.

그래서 제 대답은 (B) 40대예요.

=== 질문 1/61 | 페르소나 3/50 | 모델: google/gemma-3-12b-it ===
초기 답변: (B) 40대

아빠는 내가 어렸을 때 만났으니까, 

In [141]:
result_df2.to_csv('./multiculture_answer.csv')
result_df2.head()

,질문_ID,persona,question,model,answer
0,101801.0,"지금은 2012년 7월이다. 너는 15.0살 남성이며, 한국에서 태어나 대한민국의 수도인 서울에 15.0년 째 살고 있는 다문화 가정 중학교 2학년 학생이다. 어머니의 국적은 한국이고, 아버지 국적은 일본이다. 부모님은 현재 사별 중이며 너는 오직 아버지와 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"안녕하세요! 저는 15살 중학교 2학년 학생입니다. 아버지는... 제가 15살이니까, 아버지는 제 나이 + 30 = 45살이시겠네요. 그러니까, (B) 40대요."
1,101801.0,"지금은 2012년 7월이다. 너는 15.0살 남성이며, 한국에서 태어나 대한민국의 수도인 서울에 15.0년 째 살고 있는 다문화 가정 중학교 2학년 학생이다. 어머니의 국적은 한국이고, 아버지 국적은 일본이다. 부모님은 현재 사별 중이며 너는 오직 아버지와 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,google/gemma-3-12b-it,(C) 50대 이상\n\n아빠가... 50대 초반이신 것 같아요. 제가 2000년생이니까요. 아빠가 저를 낳으셨을 때 최소 30대 후반은 되셨을 거예요.
2,202701.0,"지금은 2012년 7월이다. 너는 14.0살 여성이며, 한국에서 태어나 대한민국의 지방에 14.0년 째 살고 있는 다문화 가정 중학교 1학년 학생이다. 어머니의 국적은 중국 (한족)이고, 아버지 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"안녕하세요! 저는 중학교 1학년 학생이에요.\n\n제 아버지는... (생각해 보면) 저가 14살이고, 제가 초등학교에 입학했을 때 아버지는 30대였을 거에요. 그럼 지금은... 40대이실 거에요.\n\n그러니까, 제 대답은 (B) 40대입니다!"
3,202701.0,"지금은 2012년 7월이다. 너는 14.0살 여성이며, 한국에서 태어나 대한민국의 지방에 14.0년 째 살고 있는 다문화 가정 중학교 1학년 학생이다. 어머니의 국적은 중국 (한족)이고, 아버지 국적은 한국이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,google/gemma-3-12b-it,(B) 40대\n\n아빠는 내가 초등학교 저학년 때 결혼하셨으니까… 지금 생각해보면 40대 초반이시겠지. 아빠도 은근히 젊어 보이고 싶어 하시니까!
4,105104.0,"지금은 2012년 7월이다. 너는 16.0살 여성이며, 한국에서 태어나 대한민국의 지방에 16.0년 째 살고 있는 다문화 가정 중학교 3학년 학생이다. 어머니의 국적은 한국이고, 아버지 국적은 일본이다. 부모님은 현재 결혼 중이며 너는 친부모 두 분과 살고 있다. 가족의 소득은 중간 정도 사는 정도이다.",당신의 아버지의 나이는?,meta-llama/Llama-4-Scout-17B-16E-Instruct,"저는 중학교 3학년이고, 올해 16살이에요. 아버지는 한국말도 잘하시고, 저희 가족과 함께 지내세요. 제 기억으로는 아버지는 저의 어머니보다 10살 이상은 더 된 것 같아요. 아마 40대쯤 되는 것 같아요.\n\n그래서 제 대답은 (B) 40대예요."
